# Redes Neurais Convolucionais (CNNs)
As CNNs remontam ao estudo do córtex visual do cérebro e têm sido usadas no reconhecimento de imagens desde a década de 1980.

## Arquitetura do córtex visual
Pesquisadores demonstraram que muitos neurônios no córtex visual têm um pequeno campo receptivo local, significando que eles reagem a estímulos visuais localizados em uma região limitada do campo visual. Os campos receptivos de diferentes neurônios podem se sobrepor e, juntos, revestir todo o campo visual.

Esses estudos do córtex visual serviram como inspiração para a neocognitron, que gradativamente evoluiu para as CNNs. O artigo de 1998 de Yann LeCun et al. apresentou a arquitetura LeNet-5, que segue alguns elementos constitutivos que já conhecemos, como camadas interligadas a funções de ativação, e a introdução de dois novos termos: camadas convolucionais e camadas de pooling.

## Camadas convolucionais
A camada convolucional é uma operação matemática que superpõe uma função sobre a outra e calcula a integral de sua multiplicação pontual (não faço ideia como isso funciona, mas é bom saber). Os neurônios na primeira camada convolucional não estão interligados a cada pixel na imagem de entrada (como estavam nas camadas analisadas nos capítulos anteriores), mas somente aos pixels em seus campos receptivos. Em contrapartida, cada neurônio na segunda camada convolucional está interligado apenas a neurônios localizados dentro de um pequeno retângulo na primeira camada.

Para que uma camada tenha a mesma altura e largura da camada anterior, é comum acrescentar zeros ao redor das entradas. Esse procedimento é conhecido como _zero-padding_. É possível interligar uma camada de entrada grande a uma camada muito menor ao espaçar os campos receptivos, o que reduz radicalmente a complexidade computacional do modelo. Isso se chama _stride_.

### Filtros
Os pesos de um neurônio podem ser representados como uma pequena imagem do tamanho do campo receptivo. Um kernel de convolução funciona como uma matriz de filtros que desliza sobre os pixels de uma imagem para extrair características fundamentais. Durante esse movimento, o kernel realiza multiplicações matemáticas entre seus valores internos e a área correspondente da imagem, somando-os para gerar um novo valor que compõe o mapa de características. Esse processo permite que a rede neural identifique elementos essenciais como bordas, texturas e formas geométricas de maneira eficiente e automatizada.

Diferentes configurações de valores dentro do kernel produzem efeitos variados no resultado final, como o realce de contornos ou o desfoque de detalhes irrelevantes. Na prática, kernels especializados em detecção de bordas conseguem isolar o esqueleto visual de um objeto, facilitando o reconhecimento de padrões mais complexos em camadas posteriores do modelo. Essa técnica é a base da visão computacional moderna, pois transforma pixels brutos em informações estruturadas e semanticamente relevantes para a inteligência artificial.

![kernel-de-convolucao.jpeg](chapter-14/images/kernel-de-convolucao.jpeg)

### Empilhando múltiplos mapas de características
Como uma camada convolucional tem diversos filtros e gera como saída um mapa de características por filtro, motivo pela qual é representada com mais precisão em 3D. Ela tem um neurônio por pixel em cada mapa de características, e todos os neurônios em um determinado mapa de características compartilham os mesmos parâmetros (pesos e vieses). Os neurônios em diferentes mapas de características usam parâmetros distintos. O campo receptivo de um neurônio é o mesmo que foi descrito anteriormente e se estende por todos os mapas de características das camadas anteriores.

Resumindo: uma camada convolucional aplica simultâneamente diversos filtros treináveis às entradas, fazendo com que elas consigam detectar características em qualquer lugar de suas entradas.

As imagens de entrada também são compostas de diversas subcamadas, uma por canal de cor (RGB). Pode ter muitos mais canais, por exemplo, imagens de satélite, que pegam também infravermelho. 


### Implementação no TensorFlow
Cada imagem é representada como um tensor 3D [height, width, channels]. Um mini-batch é representado como um tensor 4D [mini-batch size, height, width, channels]. Os pesos de uma camada convolucional são representados como um tensor de formato 4D. Os vieses como um tensor 1D.

In [ ]:
from sklearn.datasets import load_sample_image
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
keras = tf.keras

In [ ]:
china = load_sample_image('china.jpg') / 255
flower = load_sample_image('flower.jpg') / 255
images = np.array([china, flower])
batch_size, height, width, channels = images.shape

# cria dois filtros
filters = np.zeros(shape=(7,7, channels, 2), dtype=np.float32)
filters[:, 3, :, 0] = 1
filters[3, :, :, 1] = 1

# Cria dois kernels de convolução 7x7 aplicados a todos os canais RGB.
# O tensor tem formato (altura, largura, canais, nº de filtros).
# Filtro 0: ativa a coluna central → detector de bordas/linhas verticais.
# Filtro 1: ativa a linha central → detector de bordas/linhas horizontais.
# Esses filtros são exemplos manuais de feature detectors usados em CNNs.

outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.imshow(outputs[0, :, :, 1], cmap='gray')
plt.axis('off')
plt.show()

A linha `tf.nn.conv2d(images, filters, strides=1, padding='same')` aplica a operação de convolução entre o conjunto de imagens e os filtros definidos anteriormente. Cada filtro 7×7 percorre a imagem pixel a pixel (`stride = 1`), calculando o produto escalar entre os valores do kernel e os pixels da região correspondente da imagem. O parâmetro `padding='same'` adiciona preenchimento nas bordas para que a saída tenha a mesma altura e largura da imagem original.

O resultado é armazenado em `outputs`, um tensor com formato `(batch_size, altura, largura, número_de_filtros)`, onde cada filtro gera um **mapa de ativação** que indica onde o padrão aprendido pelo kernel aparece na imagem.

Na visualização, `outputs[0, :, :, 1]` seleciona o mapa de ativação do **segundo filtro** aplicado à **primeira imagem do lote**. Esse mapa é exibido com `plt.imshow(..., cmap='gray')`, onde regiões mais claras indicam locais em que o filtro respondeu com maior intensidade, ou seja, onde o padrão detectado (neste caso, linhas horizontais) está presente na imagem.

Neste exemplo definimos manualmente filtros, mas em uma CNN real normalmente definiríamos os filtros com variáveis treináveis para que a rede neural possa aprender quais filtros funcionam melhor. Em vez de criar manualmente as variáveis, use a camada `keras.layers.Conv2D`

In [ ]:
conv = keras.layers.Conv2D(filters=32, kernel_size=3, strides=1, padding='same', activation='relu')

feature_maps = conv(images)  # aplica a convolução

In [ ]:
fm = feature_maps[0, :, :, 0].numpy()
plt.imshow((fm - fm.min())/(fm.max()-fm.min()), cmap='gray') # precisa fazer normalização na visualização
plt.axis('off')
plt.show()

# Camadas de Pooling
o objetivo das camadas de pooling é subamostrar (encolhimento) a imagem de entrada para reduzir a carga computacional, o uso de memória e o número de parâmetros (restringindo o risco de overfitting).

O Neurônio de pooling não tem pesos, diferente dos neurônios convolucionais, tudo o que ele faz é agregar as entradas ao usar uma função de agregação como o máximo ou a média. Camada máxima de pooling é o tipo de camada de pooling mais comum. A operação percorre o mapa de ativação em pequenas regiões (ex.: 2×2) e seleciona apenas o maior valor dentro de cada região. 

Resumo geral até agora: A convolução detecta onde um padrão aparece. O max pooling mantém se o padrão apareceu naquela região, sem precisar da posição exata.

A camada máxima de pooling apresenta algumas desvantagens. Primeiro, ela é destrutiva, mesmo com kernel 2x2 e stride de 2, a saída será duas vezes menor em ambas as direções (a área fica quatro vezes menor) e simplesmente perde 75% dos valores de entrada.

## Implementação com TensorFlow
O padrão de strides é o tamanho do kernel, então essa camada usará um stride de 2 (tanto horizontal quanto vertical). Por padrão ele usa o padding como 'valid' (ou seja, nada de padding)

In [ ]:
max_pool = keras.layers.MaxPool2D(pool_size=2)

Para criar uma camada média de Pooling, basta usar `AvgPool2D`. A Keras não tem uma camada máxima de pooling de profundidade, mas TensorFlow tem, `tf.nn.max_pool()` e especificar o tamanho do kernel e do stride como 4-tuplas (tuplas de tamanho 4). Os três primeiros valores de cada um deve ser 1. O último valor deve ser qualquer tamanho do kernel e de stride. 

In [ ]:
output = tf.nn.max_pool(images, ksize=(1, 1, 1, 3), strides=(1, 1, 1, 3), padding='VALID')

# Caso queira incluir a camada em um dos modelos Keras, envolva em uma camada lambda

depth_pool = keras.layers.Lambda(
    lambda X: tf.nn.max_pool(images, ksize=(1, 1, 1, 3), strides=(1, 1, 1, 3), padding='VALID')
)

## Camada média de pooling global
Calcula a média do mapa inteiro de características. Isso significa que ela apenas gera a saída de um único número por mapa de características e por instância. É extremamente destrutivo (maioria das informações é perdida), pode ser útil como camada de saída

In [ ]:
global_avg_pool = keras.layers.GlobalAvgPool2D()

# Arquitetura das CNNs
Um erro comum na arquitetura de CNNs é usar kernel de convolução muito grandes. Um kernel 5x5 é a mesma coisa que dois 3x3, que faz menos cálculos e terá um melhor desempenho.

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_val = X_train_full[:5000]
X_train = X_train_full[5000:]

y_val = y_train_full[:5000]
y_train = y_train_full[5000:]

In [ ]:
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [ ]:
# Implementação de CNN com Fashion-MNIST
model = keras.models.Sequential([
    keras.layers.Conv2D(64, 7, padding='same', input_shape=[28, 28, 1]), # define 64 filtros 7x7 para extrair padrões espaciais iniciais
    keras.layers.BatchNormalization(), # normaliza as ativações do batch para estabilizar e acelerar o treinamento
    keras.layers.Activation('relu'), # aplica função de ativação não linear
    keras.layers.MaxPooling2D(2), # reduz as dimensões espaciais pela metade (28x28 → 14x14)

    keras.layers.Conv2D(128, 3, padding='same', kernel_initializer="he_normal"), # aumenta o número de filtros para capturar padrões mais complexos
    keras.layers.BatchNormalization(), # normalização para evitar instabilidade nos gradientes
    keras.layers.Activation('relu'), # ativação ReLU

    keras.layers.Conv2D(128, 3, padding='same', kernel_initializer="he_normal"), # outra convolução para refinar as features extraídas
    keras.layers.BatchNormalization(), # normalização das ativações
    keras.layers.Activation('relu'), # ativação não linear
    keras.layers.MaxPooling2D(2), # reduz novamente as dimensões espaciais (14x14 → 7x7)

    keras.layers.Conv2D(256, 3, padding='same', kernel_initializer="he_normal"), # mais filtros para detectar padrões mais abstratos
    keras.layers.BatchNormalization(), # estabiliza o treinamento
    keras.layers.Activation('relu'), # ativação ReLU

    keras.layers.Conv2D(256, 3, padding='same', kernel_initializer="he_normal"), # refina ainda mais os mapas de características
    keras.layers.BatchNormalization(), # normalização
    keras.layers.Activation('relu'), # ativação não linear

    keras.layers.Flatten(), # transforma o tensor 3D (7x7x256) em um vetor 1D para alimentar as camadas densas

    keras.layers.Dense(128), # camada totalmente conectada para combinar as features extraídas
    keras.layers.BatchNormalization(), # normaliza as ativações da camada densa
    keras.layers.Activation('relu'), # função de ativação
    keras.layers.Dropout(0.5), # desativa aleatoriamente 50% dos neurônios para reduzir overfitting

    keras.layers.Dense(64), # segunda camada densa para refinar a representação
    keras.layers.BatchNormalization(), # normalização
    keras.layers.Activation('relu'), # ativação não linear
    keras.layers.Dropout(0.5), # regularização contra overfitting

    keras.layers.Dense(10, activation='softmax'), # camada de saída com 10 classes do Fashion-MNIST
])

In [ ]:
optimizer = keras.optimizers.Adam(
    learning_rate=0.0005,
    clipvalue=1.0
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.DataFrame(history.history).plot(figsize=(8,5))
plt.grid(True)
plt.gca().set_ylim(0, 1) #precisa definir o intervalo vertical para 0 até 1
plt.show()

O modelo acima possui um overfitting visível a partir da 6a epoch. O recomendado seria testar a utilização da camada média de pooling global `global_avg_pool = keras.layers.GlobalAvgPool2D()`, ou testar a regularização $l1$ ou $l2$ antes da camada Dense fully conected: `layer = keras.layers.Dense(100, activation='elu', kernel_initializer='he_normal', kernel_regularizer=keras.regularizers.l2(0.01))`

## Arquiteturas Clássicas de Convolutional Neural Networks

As Convolutional Neural Networks (CNNs) evoluíram significativamente ao longo das últimas décadas, com diferentes arquiteturas sendo propostas para melhorar a capacidade de extração de características, eficiência computacional e profundidade dos modelos. A seguir estão alguns dos marcos mais importantes dessa evolução.

### LeNet-5

A **LeNet-5**, proposta por Yann LeCun em 1998, é uma das primeiras CNNs aplicadas com sucesso em visão computacional. Foi desenvolvida para reconhecimento de dígitos manuscritos no dataset MNIST. A arquitetura possui uma sequência simples de camadas convolucionais seguidas por camadas de pooling e camadas totalmente conectadas. As convoluções aprendem filtros que capturam padrões locais da imagem enquanto o pooling reduz a dimensionalidade e aumenta a robustez a pequenas variações espaciais. Apesar de simples pelos padrões atuais, a LeNet estabeleceu a base conceitual das CNNs modernas.

### AlexNet

A **AlexNet**, apresentada por Alex Krizhevsky, Ilya Sutskever e Geoffrey Hinton no ImageNet Challenge de 2012, marcou o início da era moderna do deep learning em visão computacional. O modelo possui oito camadas treináveis e utiliza convoluções profundas combinadas com ReLU como função de ativação, o que acelerou significativamente o treinamento. A arquitetura também popularizou o uso de GPUs para treinamento de redes neurais e introduziu técnicas como dropout para reduzir overfitting. Outro elemento importante foi o uso extensivo de *data augmentation*, que consiste em gerar novas amostras de treinamento aplicando transformações como rotações, recortes ou inversões nas imagens, aumentando artificialmente o tamanho do dataset e melhorando a capacidade de generalização do modelo.

### VGGNet

A **VGGNet**, proposta pelo Visual Geometry Group da Universidade de Oxford em 2014, buscou simplicidade estrutural. Em vez de utilizar filtros grandes, a arquitetura empilha várias convoluções pequenas de tamanho 3×3. Essa estratégia aumenta a profundidade da rede enquanto mantém um número controlado de parâmetros. O resultado é uma representação hierárquica muito poderosa das imagens. Modelos como VGG16 e VGG19 tornaram-se amplamente utilizados como redes base para transferência de aprendizado em diversos problemas de visão computacional.

### GoogLeNet

A **GoogLeNet**, também conhecida como Inception v1, foi introduzida pelo Google em 2014. Seu principal avanço é o módulo *Inception*, que aplica múltiplas convoluções com diferentes tamanhos de filtro em paralelo dentro da mesma camada. Isso permite que a rede capture padrões em múltiplas escalas simultaneamente. A arquitetura também utiliza convoluções 1×1 para reduzir dimensionalidade antes de operações mais custosas, diminuindo o número total de parâmetros e tornando a rede computacionalmente eficiente mesmo sendo muito profunda.

### ResNet

A **ResNet**, apresentada pela Microsoft em 2015, resolveu um problema fundamental das redes profundas chamado *vanishing gradient*. Seu principal conceito é o uso de *skip connections* ou conexões residuais, que permitem que o gradiente flua diretamente entre camadas distantes. Em vez de aprender diretamente uma função complexa, cada bloco aprende apenas a diferença residual em relação à entrada. Essa estratégia permite treinar redes extremamente profundas com centenas de camadas, mantendo estabilidade durante o treinamento e melhorando significativamente a performance em benchmarks de visão computacional.

### Xception

A **Xception**, proposta por François Chollet em 2017, pode ser vista como uma extensão extrema do conceito de Inception. A arquitetura substitui convoluções tradicionais por *depthwise separable convolutions*. Nesse mecanismo a convolução é separada em duas etapas: primeiro uma convolução espacial independente para cada canal e depois uma combinação linear entre canais por meio de convoluções 1×1. Essa decomposição reduz o custo computacional e melhora a eficiência da extração de características.

### SENet

A **Squeeze-and-Excitation Network (SENet)** introduz um mecanismo de atenção voltado para canais de características. O bloco *Squeeze-and-Excitation* primeiro comprime a informação espacial de cada canal para capturar seu nível global de ativação e depois aprende pesos que reescalam a importância relativa desses canais. Dessa forma a rede consegue enfatizar características mais relevantes e suprimir aquelas menos úteis para a tarefa. Essa ideia de atenção por canal pode ser incorporada a diferentes arquiteturas existentes, melhorando seu desempenho sem alterar significativamente a estrutura original.

# Implementando uma ResNet-34

In [ ]:
class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)

        self.main_layers = [
            keras.layers.Conv2D(filters, 3, strides=strides, padding="same", use_bias=False),
            keras.layers.BatchNormalization(),
            keras.layers.Activation(activation),
            keras.layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False),
            keras.layers.BatchNormalization()
        ]

        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                keras.layers.Conv2D(filters, 1, strides=strides, padding="same", use_bias=False),
                keras.layers.BatchNormalization()
            ]

    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)

        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)

        return self.activation(Z + skip_Z)

No construtor de classe acima, criamos todas as camadas necessárias, depois com o método `call()` fazemos com que as entradas passem pelas camadas principais e pelas camadas skip (se houver), em seguida adicionamos ambas as saídas e aplicamos a função de ativação.

Então podemos construir a ResNet-34 usando o modelo `Sequential`, já que temos apenas uma longa sequência de camadas (podemos tratar cada unidade residual como uma única camada agora que temos a classe acima).

In [ ]:
# Construção de um modelo sequencial no Keras
model = keras.models.Sequential([
    
    # Define o formato das imagens de entrada: 224x224 pixels com 3 canais (RGB)
    keras.Input(shape=(224, 224, 3)),
    
    # Primeira convolução da rede
    # 64 filtros 7x7 com stride 2 reduzem a resolução espacial da imagem
    # enquanto capturam padrões visuais amplos
    keras.layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False),
    
    # Batch Normalization estabiliza o treinamento normalizando ativações
    keras.layers.BatchNormalization(),
    
    # Função de ativação ReLU introduz não-linearidade
    keras.layers.Activation("relu"),
    
    # Max pooling reduz a dimensão espacial da feature map
    # mantendo apenas os valores mais relevantes em cada região
    keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same")
])

# Variável que guarda o número de filtros do bloco anterior
# usada para decidir quando mudar o stride
prev_filters = 64

# Estrutura clássica da ResNet-34
# 3 blocos com 64 filtros
# 4 blocos com 128 filtros
# 6 blocos com 256 filtros
# 3 blocos com 512 filtros
for filters in [64]*3 + [128]*4 + [256]*6 + [512]*3:
    
    # Se o número de filtros mudar, fazemos downsampling com stride 2
    # Caso contrário mantemos stride 1
    strides = 1 if filters == prev_filters else 2
    
    # Adiciona um bloco residual ao modelo
    # Cada ResidualUnit possui duas convoluções e uma skip connection
    model.add(ResidualUnit(filters, strides=strides))
    
    # Atualiza o número de filtros do bloco anterior
    prev_filters = filters

# Global Average Pooling calcula a média de cada mapa de características
# reduzindo o tensor espacial para um vetor
model.add(keras.layers.GlobalAvgPool2D())

# Flatten converte o tensor em vetor para a camada final
model.add(keras.layers.Flatten())

# Camada final de classificação
# Softmax retorna a probabilidade para cada uma das 10 classes
model.add(keras.layers.Dense(10, activation="softmax"))

In [ ]:
optimizer = keras.optimizers.Adam(
    learning_rate=0.0005,
    clipvalue=1.0
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# Usando modelos pré-treinados do Keras
Keras vem com diversas arquiteturas integradas, podemos usar os modelos disponíveis a partir de uma única linha

In [ ]:
model = keras.applications.resnet50.ResNet50(weights='imagenet') # isso fará o download dos pesos pré-treinados do conjunto de dados ImageNet

Um modelo ResNet50 espera imagens 224x224 pixels, então utilizamos a função `tf.image.resize()`

In [ ]:
images_resized = tf.image_resize(images, [224,224])

Os modelos pré-treinados partem do princípio que as imagens são pré-processadas de uma forma específica. Às vezes podem esperar que as imagens são escalonadas de 0 a 1, ou de -1 a 1, e assim por diante. Cada modelo fornece uma função `preprocess_input()`. Essas funções assumem que os valores de pixel variam de 0 a 255, então devemos multiplicar por 255

In [ ]:
inputs = keras.applications.resnet50.preprocess_input(images_resized * 250)

In [ ]:
Y_proba = model.predict(inputs)

# Modelos pré-treinados para aprendizado por transferência
se deseja construir um classificador de imagens, mas não tem dados de treinamento suficientes, podemos reutilizar as camadas inferiores de um modelo pré-treinado. Vamos treinar um modelo para classificar fotos de flores reutilizando um modelo Xception pré-treinado.

In [15]:
import tensorflow_datasets as tfds

dataset, info = tfds.load('tf_flowers', as_supervised=True, with_info=True)
dataset_size = info.splits['train'].num_examples
class_names = info.features['label'].names
n_classes = info.features['label'].num_classes

Dl Completed...: 100%|██████████| 1/1 [01:04<00:00, 64.22s/ url]
                                                                        

Dataset tf_flowers downloaded and prepared to /Users/felipetomepereira/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.


Podemos obter informações do conjunto de dados com o `with_info= True`. Como só possui o conjunto de treino ['train'], precisamos dividir o conjunto de treinamento

In [19]:
test_set = tfds.load("tf_flowers", split="train[:10%]", as_supervised=True)
valid_set = tfds.load("tf_flowers", split="train[10%:25%]", as_supervised=True)
train_set = tfds.load("tf_flowers", split="train[25%:]", as_supervised=True)

In [20]:
# Precisamos também pré-processar as imagens
def preprocess(image, label):
    resized_image = tf.image.resize(image, [224, 224])
    final_image = keras.applications.xception.preprocess_input(resized_image)
    return final_image, label

In [21]:
# Vamos aplicar a função acima para os conjuntos de dados, embaralhar, adicionar o batch e a pré-busca em todos os datasets
batch_size = 32
train_set = train_set.shuffle(100)
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1)
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1)
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1)

Caso queira usar data augmentation, use `tf.image.random_crop()` para cortar as imagens aleatóriamente, ou `tf.image.random_flip_left_right()`. A seguir vamos carregar o Xception, excluir as principais rede definindo `include_top= False`. Depois adicionamos nossa própria camada de pooling global, com base na saída do modelo básico, seguida por uma camada densa de saída com uma unidade por classe, usando softmax. Por último criamos o model da Keras

In [23]:
base_model = keras.applications.xception.Xception(weights='imagenet', include_top=False)

avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
output = keras.layers.Dense(n_classes, activation='softmax')(avg)
model = keras.Model(inputs=base_model.input, outputs= output)

In [24]:
# Geralmente é uma boa ideia congelar os pesos das camadas pré-treinadas, pelo menos no início do treinamento
for layer in base_model.layers:
    layer.trainable = False

In [26]:
# Finalmente, podemos compilar o modelo e começar o treinamento
optimizer = keras.optimizers.SGD(learning_rate=0.2, momentum=0.9, decay=0.01)
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
history = model.fit(train_set, epochs=5, validation_data=valid_set)

Epoch 1/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 32s 351ms/step - accuracy: 0.7900 - loss: 1.5518 - val_accuracy: 0.7786 - val_loss: 1.9511
Epoch 2/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 29s 341ms/step - accuracy: 0.8932 - loss: 0.7118 - val_accuracy: 0.8076 - val_loss: 1.6904
Epoch 3/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 30s 353ms/step - accuracy: 0.9121 - loss: 0.5607 - val_accuracy: 0.8312 - val_loss: 1.4875
Epoch 4/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 31s 364ms/step - accuracy: 0.9371 - loss: 0.3629 - val_accuracy: 0.8348 - val_loss: 1.3375
Epoch 5/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 36s 420ms/step - accuracy: 0.9575 - loss: 0.1932 - val_accuracy: 0.8403 - val_loss: 1.7043


Depois de treinar o modelo em algumas épocas, a acurácia de validação deve parar de progredir um pouco. Isso significa que as camadas superiores agora estão muito bem treinadas, logo, estamos prontos para descongelar todas as camadas (ou tentar descongelar apenas as camadas superiores) e continuar o treinamento (não se esqueça de compilar o modelo ao congelar ou descongelar as camadas). Desta vez, usamos uma taxa de aprendizado bem menor a fim de evitar prejudicar os pesos pré-treinados

In [27]:
for layer in base_model.layers:
    layer.trainable = False

optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, decay=0.001)
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
history = model.fit(train_set, epochs=5, validation_data=valid_set)

Epoch 1/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 31s 347ms/step - accuracy: 0.9724 - loss: 0.1180 - val_accuracy: 0.8657 - val_loss: 1.2435
Epoch 2/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 30s 352ms/step - accuracy: 0.9851 - loss: 0.0558 - val_accuracy: 0.8675 - val_loss: 1.2218
Epoch 3/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 34s 396ms/step - accuracy: 0.9880 - loss: 0.0431 - val_accuracy: 0.8657 - val_loss: 1.2113
Epoch 4/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 43s 504ms/step - accuracy: 0.9906 - loss: 0.0358 - val_accuracy: 0.8639 - val_loss: 1.2040
Epoch 5/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 46s 534ms/step - accuracy: 0.9927 - loss: 0.0311 - val_accuracy: 0.8639 - val_loss: 1.1932


# Classificação e Localização
A localização de um objeto em uma imagem pode ser traduzida em uma tarefa de regressão. Para predizer uma caixa delimitadora ao redor do objeto, uma das abordagens comuns é predizer as coordenadas horizontais e verticais do centro do objeto, bem como sua altura e largura. Isso significa 4 números para predizer. O que não exige muitas mudanças no modelo, apenas uma adicionar uma segunda camada densa de saída com quatro unidades, e ela pode ser treinada usando a função de perda MSE

In [28]:
base_model = keras.applications.xception.Xception(weights='imagenet', include_top=False)

avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
class_output = keras.layers.Dense(n_classes, activation='softmax')(avg)
loc_output = keras.layers.Dense(4)(avg)
model = keras.Model(inputs=base_model.input, outputs=[class_output, loc_output])
model.compile(loss=['sparse_categorical_crossentropy', 'mse'], loss_weights=[0.8, 0.2], optimizer=optimizer, metrics=['accuracy'])